# 5-dars: model qanday o’rganadi — xato o’lchovi va «tog’dan tushish»

4-darsda `model.fit(...)` deb yozdik va model o’rgandi. Bugun `fit` **ichida nima bo’lishini** ochamiz. Sklearn’siz, o’zimiz yozamiz.

**Bugun siz:**
- modelning «qanchalik adashdi» degan savoliga **bitta son** bilan javob berasiz (bu — **loss**);
- kompyuter xatoni kamaytirish uchun qaysi tomonga yurishni qanday biladi — shuni ko’rasiz (**qiyalik**);
- o’zingiz «o’rganadigan» dastur yozasiz (**gradient descent**) va u sklearn bilan **bir xil javob** topadi.

**Asosiy fikr:** model o’rganishi = u xatosi eng kichik bo’ladigan sonlarni qidiradi. Qidirish — ko’zi bog’langan odamning tog’dan tushishiga o’xshaydi: oyog’i bilan qiyalikni sezadi va pastga qadam tashlaydi.

**Notebook qanday ishlaydi:** matn kataklari — tushuntirish; kod kataklari ▶ (yoki Shift + Enter) bilan ishga tushadi. Kataklarni yuqoridan pastga, tartib bilan ishga tushiring.

⚠️ Avval: **Copy to Drive** (yoki **File → Save a copy in Drive**).

## 0. Kutubxonalar

Bugun yangi kutubxona — **numpy**. U sonlar ro’yxati bilan tez ishlaydi: butun ro’yxatni bir buyruqda hisoblaydi. Qolganlari tanish: `matplotlib` (grafik). `sklearn` ni faqat oxirida o’z natijamizni tekshirish uchun chaqiramiz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

## 1. Eslatma: 4-darsdagi chiziq

4-darsda 6 o’quvchi: ular necha soat o’qigan va necha ball olgan. Model shu nuqtalarga chiziq o’tkazgan edi: **ball = k × soat + b**.

- `k` — har qo’shimcha soat necha ball beradi;
- `b` — umuman o’qimasa (0 soat) necha ball bo’ladi.

Bu ikki son — modelning «ichi». Sklearn ularni topgan edi. Qanday topdi? Bugun shuni o’zimiz qilamiz.

In [ ]:
soat = np.array([1, 2, 3, 4, 6, 7])           # har o'quvchi necha soat o'qigan
ball = np.array([35, 45, 50, 65, 80, 85])     # va necha ball olgan

sk = LinearRegression().fit(soat.reshape(-1, 1), ball)    # 4-darsdagi usul
print("sklearn topgan sonlar:  k =", round(sk.coef_[0], 2), " b =", round(sk.intercept_, 2))

Sklearn javobi: **k = 8.57, b = 27.14**. Bu «to’g’ri javob». Maqsad: sklearn’siz, o’zimiz xuddi shu sonlarga yetib borish.

Avval qo’lda sinab ko’ramiz: `k` va `b` ni o’zimiz tanlaymiz va chiziq nuqtalarga qanchalik yaqin kelishini ko’ramiz.

In [ ]:
k = 5          # har soat 5 ball beradi, deb taxmin qilamiz
b = 30         # umuman o'qimasa 30 ball, deb taxmin qilamiz

taxmin = k * soat + b              # chiziq bo'yicha ball (6 ta o'quvchi uchun birdan)

plt.scatter(soat, ball, label="haqiqiy ball")            # nuqtalar — o'quvchilar
plt.plot(soat, taxmin, color="red", label="bizning chiziq")
plt.xlabel("soat"); plt.ylabel("ball"); plt.legend()
plt.show()

Chiziq nuqtalardan pastda o’tdi — biz adashdik. Lekin **qanchalik** adashdik? «Yomon» yoki «ancha yomon» demasdan, uni **songa** aylantirish kerak. Kompyuter faqat son bilan ishlaydi.

## 2. Loss: «qanchalik adashdik» degan savolga son

**Xato** = taxmin − haqiqat. Bitta o’quvchi uchun: 4 soat o’qigan, chiziq 62 ball dedi, u 65 ball olgan. Xato: 62 − 65 = **−3**.

Hamma 6 o’quvchi uchun bir buyruqda hisoblaymiz. Boshlash uchun eng sodda taxmin: «hamma 60 ball oladi» (k = 0, b = 60).

In [ ]:
taxmin = 0 * soat + 60          # k = 0, b = 60: hamma uchun 60 ball
xato = taxmin - ball            # har o'quvchi uchun: taxmin - haqiqat

print("xatolar:", xato)
print("xatolar yig'indisi:", xato.sum())

Xatolar: 25, 15, 10, −5, −20, −25. **Yig’indisi 0** chiqdi! Demak, «hamma 60» taxmini mukammal ekan? Yo’q: u yomon taxmin, faqat plyus va minus xatolar bir-birini **yo’qotdi**.

Yechim: xatoni **kvadratga ko’taramiz**. Kvadratdan keyin hamma son musbat bo’ladi, minus yo’qoladi. Katta xato ancha katta bo’lib qoladi: 5 → 25, lekin 25 → 625. So’ng o’rtachasini olamiz. Natija **MSE** (o’rtacha kvadrat xato) deyiladi. Biz uni oddiy qilib **loss** (yo’qotish) deymiz.

In [ ]:
loss = (xato ** 2).mean()       # xatolarni kvadratga ko'tar, o'rtachasini ol
print("loss =", round(loss, 1))

Loss **qancha kichik bo’lsa, model shuncha yaxshi**. Eng yaxshi holat — loss 0 (hamma taxmin aniq). Endi buni funksiyaga yig’amiz: `k` va `b` bersak, loss qaytaradi.

In [ ]:
def loss(k, b):
    taxmin = k * soat + b               # chiziq bo'yicha ball
    xato = taxmin - ball                # xatolar
    return (xato ** 2).mean()           # o'rtacha kvadrat xato

print("k=0,   b=60   ->", round(loss(0, 60), 1))        # 333.3
print("k=5,   b=30   ->", round(loss(5, 30), 1))        # 179.2
print("k=8.57, b=27.14 ->", round(loss(8.57, 27.14), 1))   # 4.8 — eng kichik

Chiziq nuqtalarga yaqinlashgan sari loss kamayadi: 333 → 179 → **4.8**. Sklearn topgan sonlarda loss eng kichik. Demak, «o’rganish» = **loss eng kichik bo’ladigan k va b ni topish**.

**Sinab ko’ring.** `k` va `b` ni qo’lda tanlab, loss ni **10 dan kichik** qiling. Qaysi sonlarni berdingiz?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

**Mustaqil ish 1.** Loss funksiyasini o’zingiz yozing. Bitta qator yetishmayapti: xato = taxmin − haqiqat.

In [ ]:
# TODO: xato qatorini to'ldiring
def loss2(k, b):
    taxmin = k * soat + b
    xato = 0                        # TODO: taxmin va haqiqiy ballning farqi
    return np.mean(xato ** 2)

natija1 = loss2(0, 60)
assert abs(natija1 - 333.3) < 0.1, f"loss2(0, 60) = {natija1}, lekin 333.3 chiqishi kerak. xato = taxmin - ball yozdingizmi?"
print(round(natija1, 1))            # 333.3 chiqishi kerak

## 3. Avval faqat bitta son: `k` ni topamiz

Bir vaqtda ikkita son qidirish qiyinroq. Shuning uchun avval `b` ni qotirib qo’yamiz (`b = 27`), faqat `k` ni qidiramiz. Endi loss bitta songa bog’liq: `k` ni o’zgartirsak, loss ham o’zgaradi. Buni grafikda ko’ramiz.

In [ ]:
def loss_k(k):
    return loss(k, 27)          # b = 27 qotirilgan, faqat k o'zgaradi

k_lar = np.linspace(0, 12, 100)                  # k uchun 100 ta qiymat: 0 dan 12 gacha
loss_lar = [loss_k(k) for k in k_lar]            # har biri uchun loss

plt.plot(k_lar, loss_lar)
plt.xlabel("k (har soat necha ball)"); plt.ylabel("loss")
plt.title("k o'zgarsa, loss qanday o'zgaradi")
plt.show()

Grafik **vodiy** shaklida. Chap tomonda `k` kichik, loss katta (chiziq juda yotiq). O’ngda `k` katta, loss yana katta (chiziq juda tik). Eng pastki nuqta — eng yaxshi `k`, taxminan **8.6**.

Shu pastki nuqtani qanday topamiz? Eng oddiy yo’l — hamma qiymatni sinab, eng kichigini olish.

In [ ]:
sinaladigan = np.linspace(0, 12, 121)                    # 0, 0.1, 0.2, ... 12
losslar = [loss_k(k) for k in sinaladigan]               # hammasi uchun loss
eng_yaxshi = sinaladigan[np.argmin(losslar)]             # eng kichik loss qaysi joyda
print("eng yaxshi k:", round(eng_yaxshi, 1), "  ", len(sinaladigan), "ta sinov")

Ishladi: **k = 8.6**, lekin 121 ta sinov bilan. Bu usul bizni yarim yo’lda tashlaydi:
- ikkita son (`k` va `b`) bo’lsa, juftliklar soni 121 × 121 ≈ 14 mingga chiqadi;
- real modelda sonlar **millionlab** (ChatGPT’da — yuz milliardlab). Hammasini sinab ko’rish mumkin emas — dunyo tugaguncha ham yetmaydi.

Aqlliroq yo’l kerak: nuqtani **ko’r-ko’rona emas, yo’nalishni bilib** qidirish.

## 4. Qiyalik: qaysi tomonga yurish kerak?

Tasavvur qiling: siz tog’ yonbag’rida **tuman ichida** turibsiz, hech narsa ko’rinmaydi. Pastga tushish kerak. Nima qilasiz? Oyog’ingiz bilan **yer qaysi tomonga qiyaligini** sezasiz va pastga qarab qadam tashlaysiz.

Bizda tog’ — loss grafigi. Biz turgan joy — hozirgi `k`. Qiyalikni qanday sezamiz? **Kichik qadam bilan**: `k` ni ozgina (0.001) surib, loss qancha o’zgarganini ko’ramiz.

qiyalik = (loss(k + 0.001) − loss(k)) / 0.001

In [ ]:
k = 4
h = 0.001                                         # juda kichik qadam

oldin = loss_k(k)                                 # hozirgi joyda loss
keyin = loss_k(k + h)                             # ozgina o'ngga surgandan keyin loss
qiyalik = (keyin - oldin) / h                     # o'zgarish / surilish

print("loss oldin:", round(oldin, 2), " keyin:", round(keyin, 2))
print("qiyalik:", round(qiyalik, 1))

Qiyalik **−176**, ya’ni **manfiy**: `k` ni oshirsak, loss **kamayadi**. Demak, pastga tushish uchun `k` ni oshirish kerak. Qoida:

| Qiyalik | Ma’nosi | Nima qilamiz |
|---|---|---|
| manfiy (−) | k ni oshirsak, loss kamayadi | k ni **oshiramiz** |
| musbat (+) | k ni oshirsak, loss o’sadi | k ni **kamaytiramiz** |
| 0 ga yaqin | vodiyning tubi | **to’xtaymiz** |

Qiyalik kattaligi ham ma’lumot beradi: tik joyda (katta son) pastlik uzoq, katta qadam tashlasa bo’ladi. Tekis joyda (kichik son) pastlikka yaqinmiz.

**Sinab ko’ring.** `k = 12` da qiyalikni hisoblang. Avval taxmin qiling: ishorasi qanday bo’ladi? `k` ni oshirish kerakmi yoki kamaytirish?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Qiyalikni har safar yozmaslik uchun funksiyaga aylantiramiz. Bu funksiya **har qanday** loss uchun ishlaydi (keyin ikkinchi son uchun ham kerak bo’ladi).

**Mustaqil ish 2.** `qiyalik(f, x)` funksiyasini yozing: `f` — istalgan funksiya, `x` — qaysi joyda qiyalikni o’lchaymiz.

In [ ]:
# TODO: qiyalik formulasini yozing
def qiyalik(f, x):
    h = 0.001
    return 0                # TODO: (f(x + h) - f(x)) / h

# tekshiruv: x*x funksiyasining x=3 dagi qiyaligi 6 ga yaqin
tekshir = qiyalik(lambda t: t * t, 3)
assert abs(tekshir - 6) < 0.01, f"qiyalik(x*x, 3) = {tekshir}, lekin 6 chiqishi kerak. (f(x + h) - f(x)) / h yozdingizmi?"
print(round(tekshir, 2))    # 6.0 chiqishi kerak

## 5. Gradient descent: qadam-baqadam pastga

Endi hammasini birlashtiramiz. Dastur shunday ishlaydi:

1. Istalgan joydan boshlaymiz (masalan, `k = 0`).
2. **Qiyalikni** o’lchaymiz.
3. Qiyalikka **teskari** tomonga qadam tashlaymiz: `k = k − lr × qiyalik`.
4. 2–3 ni takrorlaymiz. Qiyalik 0 ga yaqinlashganda — vodiyning tubidamiz.

`lr` (**learning rate**, «o’rganish tezligi») — qadam kattaligi. Bu usulning nomi — **gradient descent** (gradient = qiyalik, descent = tushish). Mana, bu `fit` ichida bo’ladigan ish. Birinchi 10 qadamni jadvalda ko’ramiz.

In [ ]:
k = 0.0                    # boshlanish: tog'ning istalgan joyi
lr = 0.01                  # qadam kattaligi

for qadam in range(10):
    qiy = qiyalik(loss_k, k)                                   # qiyalikni o'lcha
    print(f"{qadam}-qadam: k = {k:.2f}   qiyalik = {qiy:7.1f}   loss = {loss_k(k):7.1f}")
    k = k - lr * qiy                                           # qiyalikka teskari tomonga yur

Nima ko’rinyapti:
- `k` har qadamda **8.6 ga yaqinlashyapti**: 0 → 3.3 → 5.3 → 6.6 → …;
- `loss` **kamayyapti**: 1422 → 544 → 210 → … → 5;
- qadamlar **kichrayyapti** (3.3, 2.0, 1.3, 0.8…): sababi qiyalik kichrayyapti, ya’ni vodiy tubiga yaqinlashyapmiz. Dastur o’zi sekinlashadi.

**Mustaqil ish 3.** Xuddi shu tsiklni o’zingiz yozing. Qadam kattaligi bu safar `lr = 0.02`, 30 qadam. Yetishmayotgan qator — `k` ni yangilash.

In [ ]:
# TODO: k ni yangilash qatorini yozing
k = 0.0
lr = 0.02

for qadam in range(30):
    qiy = qiyalik(loss_k, k)
    k = k                   # TODO: k = k - lr * qiy

assert abs(k - 8.6) < 0.1, f"k = {k:.2f} chiqdi, lekin 8.6 ga yaqin bo'lishi kerak. k = k - lr * qiy yozdingizmi?"
print(round(k, 2))          # 8.6 chiqishi kerak

Dastur **o’zi** k = 8.6 ni topdi, hech qanday formula bilmasdan. 121 ta sinov o’rniga atigi bir necha qadam. Aynan shu g’oya ChatGPT’ni ham o’qitadi, faqat u yerda `k` o’rniga milliardlab son.

## 6. Learning rate: qadam qanchalik katta bo’lsin?

`lr` — eng muhim sozlamalardan biri. Uch holatni solishtiramiz:
- **juda kichik** (`0.001`): ehtiyotkor, lekin juda sekin;
- **yaxshi** (`0.01`): tez va tinch;
- **juda katta** (`0.06`): har qadam vodiy bo’ylab sakrab, narigi devorga tushadi.

Har lr uchun 15 qadam davomida loss qanday o’zgarishini chizamiz.

In [ ]:
def qadamlar(lr, n=15):
    k = 0.0
    royxat = [loss_k(k)]                       # loss tarixi
    for i in range(n):
        k = k - lr * qiyalik(loss_k, k)
        royxat.append(loss_k(k))
    return royxat

for lr, nom in [(0.001, "juda kichik (0.001)"), (0.01, "yaxshi (0.01)"), (0.06, "juda katta (0.06)")]:
    plt.plot(qadamlar(lr), label=nom)

plt.ylim(0, 3000)                              # juda katta lr loss'i osmonga chiqib ketadi, grafikni kesamiz
plt.xlabel("qadam"); plt.ylabel("loss"); plt.legend()
plt.title("Learning rate'ning ta'siri")
plt.show()

- **0.001**: loss kamayyapti, lekin 15 qadamda ham vodiy tubiga yetmadi;
- **0.01**: bir necha qadamda loss deyarli 0 ga tushdi;
- **0.06**: loss **kamaymayapti, aksincha o’syapti**. Har qadam shunchalik katta, narigi devorga sakrab tushadi, u yerda qiyalik yana katta, keyingi sakrash yanada katta. Dastur «portlaydi».

Shuning uchun `lr` ni tanlash — mashinali o’rganishdagi haqiqiy mashq: juda kichik bo’lsa vaqt ketadi, juda katta bo’lsa o’rganmaydi.

**Sinab ko’ring.** `lr = 0.05` bilan 8 qadam bajaring va `k` qanday o’zgarishini yozib chiqing. Avval taxmin qiling: tinch tushadimi yoki sakraydimi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 7. Ikkala son: `k` va `b`

Hozirgacha `b` ni qotirgan edik. Haqiqiy model ikkalasini bir vaqtda qidiradi. G’oya o’zgarmaydi, faqat tog’ endi **ikki yo’nalishli**: `k` bo’yicha ham, `b` bo’yicha ham qiyalikni o’lchaymiz va ikkalasini birdan yangilaymiz:

- `qiy_k` — `k` ni ozgina surib, loss qancha o’zgardi;
- `qiy_b` — `b` ni ozgina surib, loss qancha o’zgardi;
- `k = k − lr × qiy_k`, `b = b − lr × qiy_b`.

Hammasini bitta funksiyaga yig’amiz: u istalgan `x`, `y` ma’lumot uchun `k` va `b` ni topadi. Har qadamni `tarix` ro’yxatida saqlaymiz, keyin chizamiz.

In [ ]:
def o_rgat(x, y, lr=0.02, n=2000):
    def loss(k, b):                                  # shu x, y uchun loss
        return ((k * x + b - y) ** 2).mean()

    k, b = 0.0, 0.0                                  # boshlanish: ikkalasi 0
    h = 0.001                                        # qiyalik o'lchaydigan kichik qadam
    tarix = []                                       # har qadamdagi (k, b, loss)

    for i in range(n + 1):
        tarix.append((k, b, loss(k, b)))
        qiy_k = (loss(k + h, b) - loss(k, b)) / h    # k bo'yicha qiyalik
        qiy_b = (loss(k, b + h) - loss(k, b)) / h    # b bo'yicha qiyalik
        k = k - lr * qiy_k                           # ikkalasini birdan yangilaymiz
        b = b - lr * qiy_b
    return k, b, tarix

Funksiya tayyor. Uni 6 o’quvchi ma’lumotida ishga tushiramiz va natijani sklearn bilan solishtiramiz.

In [ ]:
k, b, tarix = o_rgat(soat, ball)          # 2000 qadam, lr = 0.02

print("biz topdik:    k =", round(k, 2), "  b =", round(b, 2))
print("sklearn topdi: k =", round(sk.coef_[0], 2), "  b =", round(sk.intercept_, 2))

**Bir xil javob: k = 8.57, b = 27.15.** Biz sklearn’siz, faqat «kichik qadam bilan qiyalikni o’lchab, pastga tushish» yo’li bilan o’zimiz `fit` yozdik. Endi `fit` sirli quti emas.

Bu jarayonni ko’zimiz bilan ko’ramiz: har qadamda chiziq qanday o’zgargan? 0, 1, 5, 20, 100 va 2000-qadamdagi chiziqlarni bir rasmga chizamiz.

In [ ]:
for qadam in [0, 1, 5, 20, 100, 2000]:
    kk, bb, ll = tarix[qadam]
    plt.plot(soat, kk * soat + bb, label=f"{qadam}-qadam (loss {ll:.0f})")    # shu qadamdagi chiziq

plt.scatter(soat, ball, color="black", zorder=5)                              # haqiqiy nuqtalar
plt.xlabel("soat"); plt.ylabel("ball"); plt.legend()
plt.title("Chiziq qadam-baqadam nuqtalarga yaqinlashadi")
plt.show()

Birinchi chiziq (0-qadam) — tekis nol. 1-qadamdayoq tik ko’tarildi, 20-qadamda nuqtalarga yaqinlashdi, 100-qadamdan keyin deyarli oxirgi chiziq bilan bir joyda.

Endi shu o’rganish jarayonining **loss grafigi**: loss har qadamda kamayishi kerak.

In [ ]:
losslar = [t[2] for t in tarix[:200]]         # birinchi 200 qadamdagi loss

plt.plot(losslar)
plt.xlabel("qadam"); plt.ylabel("loss")
plt.title("O'rganish jarayoni: loss kamayadi")
plt.show()

Avval loss **tez** tushadi (tik joy), keyin **sekin** (tekislik). Bu tabiiy: tubga yaqinlashgan sari qiyalik kichrayadi, qadamlar ham kichrayadi. Haqiqiy modellarni o’qitganda ham aynan shu grafikni kuzatishadi: loss tushyaptimi, to’xtab qoldimi, sakrayaptimi.

**Sinab ko’ring.** `lr = 0.05` bilan 100 qadam o’qiting va oxirgi loss’ni chiqaring. Avval taxmin qiling: 4.8 ga yetadimi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 8. Mustaqil ish: yangi ma’lumotda sinaymiz

Funksiya `o_rgat` **har qanday** `x` va `y` bilan ishlaydi. Yangi ma’lumot: kichkina do’kon 5 kun davomida necha ming so’m pul yig’gan.

**Mustaqil ish 4.**
1. `o_rgat(kun, pul)` ni chaqiring;
2. natijani sklearn bilan solishtiring. Sonlar bir xil chiqishi kerak.

In [ ]:
kun = np.array([1, 2, 3, 4, 5])             # kun raqami
pul = np.array([12, 19, 31, 38, 52])        # shu kuni yig'ilgan pul (ming so'm)

# TODO: o_rgat funksiyasini kun va pul bilan chaqiring
natija4 = ...

assert natija4 is not ..., "natija4 = ... o'rniga o_rgat(kun, pul) ni yozing"
k2, b2, tarix2 = natija4

sk2 = LinearRegression().fit(kun.reshape(-1, 1), pul)
print("biz topdik:    k =", round(k2, 1), "  b =", round(b2, 1))            # 9.9 va 0.7
print("sklearn topdi: k =", round(sk2.coef_[0], 1), "  b =", round(sk2.intercept_, 1))

Natija: **k ≈ 9.9, b ≈ 0.7**. Ya’ni do’kon har kuni o’rtacha ~9.9 ming so’m ko’proq yig’adi. Yangi ma’lumotda ham o’z dasturimiz sklearn bilan bir xil javob berdi.

## 9. Bu nega muhim?

Bugun ikkita son topdik. Bu g’oya hamma joyda:
- **neyron tarmoq** (6-dars): minglab sonlar, xuddi shu «qiyalikni o’lcha va pastga tush» qoidasi;
- **ChatGPT**: yuz milliardlab son, xuddi shu qoida, faqat qiyalikni tez hisoblash uchun maxsus usul (**backpropagation**, 6-darsda).

Demak, mashinali o’rganishning yuragi:
1. modelni **loss** bilan baholash;
2. loss’ni **kamaytirish** uchun sonlarni **kichik qadam** bilan o’zgartirish;
3. loss kamayib to’xtaguncha **takrorlash**.

`model.fit(...)` yozganingizda sklearn aynan shu ishni qiladi.

## Uy ishi: learning rate’ni sinab ko’ring

`o_rgat` funksiyasini 6 o’quvchi ma’lumotida **4 xil `lr`** bilan 100 qadam ishga tushiring: `0.001`, `0.01`, `0.03`, `0.05`. Har biri uchun loss grafigini bitta rasmga chizing va quyidagilarga **raqamlar bilan** javob yozing:
1. Qaysi `lr` 100 qadamda eng past loss’ga yetdi? Qaysi biri hali o’rganib bo’lmadi?
2. Qaysi `lr` sakradi (loss ko’tarilib-tushdi) yoki o’rgana olmadi?
3. Nega «eng katta lr eng tez» degan qoida noto’g’ri?

Boshlash uchun shablon:

In [ ]:
for lr in [0.001, 0.01, 0.03, 0.05]:
    kk, bb, tt = o_rgat(soat, ball, lr=lr, n=100)
    plt.plot([t[2] for t in tt], label=f"lr = {lr}")        # loss grafigi

plt.ylim(0, 500)
plt.xlabel("qadam"); plt.ylabel("loss"); plt.legend()
plt.show()

**Javobingiz:** shu katakni ikki marta bosib yozing.

## Chuqurroq (ixtiyoriy): qiyalikni formula bilan

Biz qiyalikni **kichik qadam** bilan o’lchadik (qo’pol, lekin sodda). Matematikada buning aniq formulasi bor (hosila). `loss = o’rtacha((k·x + b − y)²)` uchun:

- `k` bo’yicha qiyalik = o’rtacha( 2 · xato · soat )
- `b` bo’yicha qiyalik = o’rtacha( 2 · xato )

Tekshiramiz: `k = 0`, `b = 0` nuqtada ikkala usul bir xil javob beradimi?

In [ ]:
k, b = 0.0, 0.0
h = 0.001

# 1) kichik qadam bilan (biz ishlatgan usul)
q_k = (loss(k + h, b) - loss(k, b)) / h
q_b = (loss(k, b + h) - loss(k, b)) / h

# 2) formula bilan
xato = k * soat + b - ball
f_k = np.mean(2 * xato * soat)
f_b = np.mean(2 * xato)

print("kichik qadam:", round(q_k, 1), round(q_b, 1))     # -536.6 va -120.0
print("formula:     ", round(f_k, 1), round(f_b, 1))     # -536.7 va -120.0

Deyarli bir xil. Haqiqiy kutubxonalar (PyTorch) formulani avtomatik chiqaradi, shu sababli juda tez ishlaydi. G’oya esa bugun ko’rganimizdek: qiyalik → pastga qadam.

## Yakun

Bugun siz:
- modelning xatosini bitta son (**loss, MSE**) bilan o’lchadingiz;
- kichik qadam bilan **qiyalikni** o’lchab, qaysi tomonga yurishni bildingiz;
- **gradient descent** yozdingiz: `k = k − lr × qiyalik`;
- **learning rate** juda kichik bo’lsa sekin, juda katta bo’lsa portlashini ko’rdingiz;
- o’zingiz yozgan dastur sklearn bilan **bir xil javob** topdi: `fit` ichida shu ish bo’ladi.

Keyingi darsda (6-dars) bir `k` va `b` o’rniga **ko’p sonli** model yasaymiz: neyron tarmoq. U ham xuddi shunday o’qitiladi.